# 模拟退火算法（Simulated Annealing, SA）

模拟退火算法是一种随机优化算法，灵感来自金属退火过程。金属在高温下，内部粒子运动剧烈，可以跳出当前结构；随着温度慢慢降低，粒子逐渐稳定，最终形成能量较低、结构较稳定的状态。

把这个思想放到优化问题里：

- 当前解相当于材料当前状态；
- 目标函数值相当于系统能量；
- 寻找最优解相当于寻找最低能量状态；
- 高温时允许较大随机搜索；
- 低温时逐渐收敛到较优解。

一句话理解：模拟退火不是每一步都只接受更好的解，它有时会故意接受更差的解，从而跳出局部最优。

## 1. 适用问题

模拟退火适合处理：

- 非线性优化问题；
- 多峰函数优化问题；
- 组合优化问题，如旅行商问题、排产调度、路径规划；
- 目标函数不可导或难以求梯度的问题；
- 容易陷入局部最优的问题。

它的优点是思想简单、实现灵活；缺点是参数和降温策略会明显影响效率。

## 2. 优化问题形式

以最小化问题为例：

$$
\min_{x \in \Omega} f(x)
$$

其中：

- $x$ 是当前解；
- $f(x)$ 是目标函数；
- $\Omega$ 是可行域。

模拟退火每次从当前解 $x$ 出发，随机生成一个邻域解 $x'$，再决定是否接受这个新解。

## 3. Metropolis 接受准则

设当前解为 $x$，新解为 $x'$，目标函数变化量为：

$$
\Delta f = f(x') - f(x)
$$

对于最小化问题：

- 如果 $\Delta f \le 0$，说明新解更好或相同，直接接受；
- 如果 $\Delta f > 0$，说明新解更差，不是直接拒绝，而是按概率接受。

接受更差解的概率为：

$$
P = \exp\left(-\frac{\Delta f}{T}\right)
$$

其中 $T$ 是当前温度。

这个公式很关键：

- 温度 $T$ 高时，接受差解的概率较大，有利于全局搜索；
- 温度 $T$ 低时，接受差解的概率较小，有利于局部收敛；
- 差得越多，即 $\Delta f$ 越大，被接受的概率越小。

## 4. 降温策略

模拟退火需要从高温逐渐降到低温。常见降温方式包括：

### 4.1 几何降温

$$
T_{k+1} = \alpha T_k, \quad 0 < \alpha < 1
$$

这是最常用的方式，通常 $\alpha$ 取 $0.90$ 到 $0.99$。

### 4.2 线性降温

$$
T_{k+1} = T_k - \beta
$$

实现简单，但温度可能过快降到 0。

### 4.3 对数降温

$$
T_k = \frac{T_0}{\ln(1+k)}
$$

理论性质较好，但实际运行往往较慢。

建模和编程中最常用的是几何降温。

## 5. 算法流程

模拟退火算法的一般流程如下：

1. 随机生成或指定一个初始解 $x$。
2. 设置初始温度 $T_0$、终止温度 $T_{min}$、降温系数 $\alpha$。
3. 在当前温度下重复若干次邻域搜索：
   - 由当前解生成邻域解 $x'$；
   - 计算 $\Delta f = f(x') - f(x)$；
   - 根据 Metropolis 准则决定是否接受 $x'$；
   - 记录历史最优解。
4. 降低温度。
5. 当温度低于终止温度或达到最大迭代次数时停止。
6. 输出历史最优解。

In [ ]:
import numpy as np

try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False
except ModuleNotFoundError:
    plt = None
    print("当前环境没有安装 matplotlib，绘图单元会跳过。可运行：pip install matplotlib")


## 6. 从零实现连续变量模拟退火

下面实现一个适合连续变量最小化问题的模拟退火算法。邻域解通过对当前解加入随机扰动产生。

In [ ]:
def simulated_annealing_minimize(
    func,
    bounds,
    initial_temp=100.0,
    final_temp=1e-3,
    cooling_rate=0.95,
    steps_per_temp=100,
    step_scale=0.1,
    seed=42,
):
    """使用模拟退火算法最小化连续变量目标函数。

    Parameters
    ----------
    func : callable
        目标函数，输入一维数组 x，返回标量函数值。
    bounds : list[tuple[float, float]]
        每个变量的上下界，例如 [(-5, 5), (-5, 5)]。
    initial_temp : float
        初始温度。
    final_temp : float
        终止温度。
    cooling_rate : float
        几何降温系数，通常取 0.90 到 0.99。
    steps_per_temp : int
        每个温度下尝试生成邻域解的次数。
    step_scale : float
        邻域扰动强度，占变量范围的比例。
    seed : int
        随机种子，便于复现实验。
    """
    rng = np.random.default_rng(seed)
    bounds = np.asarray(bounds, dtype=float)
    lower = bounds[:, 0]
    upper = bounds[:, 1]
    span = upper - lower

    current_x = rng.uniform(lower, upper)
    current_y = func(current_x)
    best_x = current_x.copy()
    best_y = current_y

    temp = initial_temp
    history = [best_y]
    temp_history = [temp]
    accepted_count = 0
    total_count = 0

    while temp > final_temp:
        for _ in range(steps_per_temp):
            candidate_x = current_x + rng.normal(0, step_scale * span)
            candidate_x = np.clip(candidate_x, lower, upper)
            candidate_y = func(candidate_x)

            delta = candidate_y - current_y
            accept = delta <= 0 or rng.random() < np.exp(-delta / temp)

            total_count += 1
            if accept:
                current_x = candidate_x
                current_y = candidate_y
                accepted_count += 1

                if current_y < best_y:
                    best_x = current_x.copy()
                    best_y = current_y

            history.append(best_y)
            temp_history.append(temp)

        temp *= cooling_rate

    acceptance_rate = accepted_count / total_count
    return best_x, best_y, np.array(history), np.array(temp_history), acceptance_rate


## 7. 示例一：Sphere 函数

Sphere 函数为：

$$
f(x) = \sum_{j=1}^{d} x_j^2
$$

它的全局最优解为 $x=(0,0,\dots,0)$，最优值为 $0$。

In [ ]:
def sphere_single(x):
    return np.sum(x ** 2)


best_x, best_y, history, temp_history, acceptance_rate = simulated_annealing_minimize(
    sphere_single,
    bounds=[(-5, 5), (-5, 5)],
    initial_temp=20,
    final_temp=1e-4,
    cooling_rate=0.94,
    steps_per_temp=80,
    step_scale=0.08,
    seed=1,
)

print("最优位置：", best_x)
print("最优函数值：", best_y)
print("接受率：", acceptance_rate)


In [ ]:
if plt is None:
    print("跳过绘图：请先安装 matplotlib。")
else:
    plt.figure(figsize=(7, 4))
    plt.plot(history, linewidth=2)
    plt.xlabel("搜索步数")
    plt.ylabel("历史最优目标函数值")
    plt.title("模拟退火在 Sphere 函数上的收敛曲线")
    plt.grid(alpha=0.3)
    plt.show()


## 8. 示例二：Rastrigin 函数

Rastrigin 函数是多峰函数：

$$
f(x) = 10d + \sum_{j=1}^{d}\left[x_j^2 - 10\cos(2\pi x_j)\right]
$$

全局最优解为 $x=(0,0,\dots,0)$，最优值为 $0$。由于局部最优点很多，它很适合用来观察模拟退火“允许接受差解”的意义。

In [ ]:
def rastrigin_single(x):
    dim = len(x)
    return 10 * dim + np.sum(x ** 2 - 10 * np.cos(2 * np.pi * x))


best_x, best_y, history, temp_history, acceptance_rate = simulated_annealing_minimize(
    rastrigin_single,
    bounds=[(-5.12, 5.12), (-5.12, 5.12)],
    initial_temp=50,
    final_temp=1e-4,
    cooling_rate=0.96,
    steps_per_temp=120,
    step_scale=0.06,
    seed=2,
)

print("最优位置：", best_x)
print("最优函数值：", best_y)
print("接受率：", acceptance_rate)


In [ ]:
if plt is None:
    print("跳过绘图：请先安装 matplotlib。")
else:
    plt.figure(figsize=(7, 4))
    plt.semilogy(history + 1e-12, linewidth=2)
    plt.xlabel("搜索步数")
    plt.ylabel("历史最优目标函数值（对数坐标）")
    plt.title("模拟退火在 Rastrigin 函数上的收敛曲线")
    plt.grid(alpha=0.3)
    plt.show()


## 9. 参数解释与调参建议

| 参数 | 含义 | 常见设置 | 调参建议 |
| --- | --- | --- | --- |
| 初始温度 $T_0$ | 初期接受差解的能力 | 10 到 1000 | 初期太保守就增大 |
| 终止温度 $T_{min}$ | 停止搜索的温度 | $10^{-5}$ 到 $10^{-2}$ | 要更精细可减小 |
| 降温系数 $\alpha$ | 温度下降速度 | 0.90 到 0.99 | 越接近 1 搜索越充分但越慢 |
| 每温度搜索次数 | 每个温度下尝试邻域解次数 | 50 到 500 | 复杂问题适当增加 |
| 邻域步长 | 新解扰动幅度 | 变量范围的 1% 到 20% | 太大难收敛，太小难跳出局部最优 |

实用建议：

- 如果结果很不稳定，可以增加每个温度下的搜索次数；
- 如果算法很快停住，可以提高初始温度或减慢降温速度；
- 如果后期一直震荡，可以减小邻域步长；
- 模拟退火是随机算法，建议多运行几次比较结果。

## 10. 约束问题处理

模拟退火处理约束问题时，常用方法包括：

- 边界裁剪：变量超出上下界时拉回可行范围；
- 拒绝不可行解：若新解不满足约束，直接拒绝；
- 罚函数法：违反约束时增加惩罚项；
- 修复法：把不可行解修正为可行解。

罚函数法示例：

$$
\min f(x), \quad g(x) \le 0
$$

可以构造：

$$
F(x) = f(x) + M\max(0, g(x))^2
$$

其中 $M$ 是罚因子。$M$ 太小会导致约束不够严格，太大可能让搜索变得困难。

## 11. 与遗传算法、粒子群算法的比较

| 方面 | 模拟退火 SA | 遗传算法 GA | 粒子群优化 PSO |
| --- | --- | --- | --- |
| 搜索对象 | 单个当前解 | 一组个体 | 一群粒子 |
| 核心机制 | 概率接受差解 | 选择、交叉、变异 | 个体经验和群体经验 |
| 参数数量 | 较少 | 较多 | 中等 |
| 全局搜索能力 | 依赖温度和降温 | 依赖种群多样性 | 依赖粒子分布 |
| 实现难度 | 简单 | 中等 | 简单 |
| 典型应用 | 连续优化、组合优化 | 组合优化、参数优化 | 连续参数优化 |

简单选择建议：

- 如果想要一个结构简单、容易嵌入其他模型的随机优化方法，可以选模拟退火；
- 如果问题是路径、排序、调度等组合优化，GA 和 SA 都常用；
- 如果问题是连续参数寻优，PSO 往往收敛更快，但 SA 的跳出局部最优思想很有价值。

## 12. 数学建模中的写作模板

在论文或报告中介绍模拟退火算法，可以按下面结构写：

1. 解的表示：说明一个解 $x$ 如何对应实际问题方案。
2. 目标函数：说明 $f(x)$ 如何计算，若有约束则说明罚函数。
3. 邻域结构：说明如何从当前解生成新解。
4. 接受准则：写出 Metropolis 接受概率。
5. 降温策略：说明初始温度、终止温度、降温系数。
6. 终止条件：如温度低于阈值或达到最大迭代次数。
7. 结果分析：给出最优解、目标函数值、收敛曲线和参数敏感性分析。

## 13. 小结

模拟退火的优点：

- 原理简单，代码实现容易；
- 不需要目标函数可导；
- 能以一定概率跳出局部最优；
- 连续优化和组合优化都能使用。

模拟退火的缺点：

- 收敛速度可能较慢；
- 参数设置影响明显；
- 邻域结构设计很重要；
- 单次运行结果可能有随机波动。

一句话记忆：模拟退火靠“高温敢乱走、低温慢慢稳”的策略，在搜索过程中既能探索全局，也能逐渐收敛到较好的解。